# User-facing control that affects LLM Experimentation
(e.g., a slider for response verbosity, a dropdown to switch response style, or a toggle to restrict scope).

Your repository should contain a notebook with experiments and narrative, and the resulting option and summary of motivation should be reflected in the specifications document.

## Imports

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from chatlas import ChatAnthropic

## Load, merge, and view data

In [4]:
FEATURES_PATH = Path("../data/raw/ai_productivity_features.csv")
TARGETS_PATH = Path("../data/raw/ai_productivity_targets.csv")

features = pd.read_csv(FEATURES_PATH)
targets = pd.read_csv(TARGETS_PATH)

df = features.merge(targets, on="Employee_ID", how="left")
df.head()

,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,burnout_risk_score,productivity_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,10.00,81.0,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,6.78,59.2,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,9.66,62.4,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,10.00,76.8,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,5.38,53.7,Medium


## Define the dataset description

In [5]:
data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- learning_time_hours_per_week: hours per week spent learning new tools/workflows.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- error_rate_percent: percentage error rate.
- task_complexity_score: task complexity score.
- focus_hours_per_day: average focus/deep work hours per day.
- work_life_balance_score: numeric work-life balance score.
- burnout_risk_score: numeric burnout risk score.
- productivity_score: numeric productivity score.
- burnout_risk_level: burnout category label.

Use this dataset to analyze:
- burnout risk by role
- relationships between AI usage and burnout
- productivity vs burnout
- work-life balance patterns
- workload-related drivers of burnout

Important:
- Stay grounded in the available dataset columns.
- Do not make causal claims.
- Frame findings as associations, patterns, or comparisons.
"""

## Create a compact dataset summary for the prompt

This helps the model stay grounded.

In [ ]:
def build_dataset_snapshot(df: pd.DataFrame) -> str:
    role_counts = df["job_role"].value_counts().to_dict()
    burnout_counts = df["burnout_risk_level"].value_counts().to_dict()

    summary = {
        "n_rows": int(len(df)),
        "job_roles": role_counts,
        "burnout_levels": burnout_counts,
        "avg_ai_usage_hours": round(
            df["ai_tool_usage_hours_per_week"].mean(),
            2
        ),
        "avg_manual_hours": round(
            df["manual_work_hours_per_week"].mean(), 2),
        "avg_burnout_score": round(
            df["burnout_risk_score"].mean(), 
            2
        ),
        "avg_productivity_score": round(df["productivity_score"].mean(), 2),
        "avg_work_life_balance": round(
            df["work_life_balance_score"].mean(),
            2
        ),
    }

    lines = ["Dataset snapshot:"]
    for k, v in summary.items():
        lines.append(f"- {k}: {v}")
    return "\n".join(lines)

dataset_snapshot = build_dataset_snapshot(df)
print(dataset_snapshot)

## Define the style instructions

In [ ]:
STYLE_INSTRUCTIONS = {
    "executive": 
    """
    Respond in an Executive Summary style.
    - Use plain, concise language.
    - Emphasize the main takeaway first.
    - Mention practical workplace implications.
    - Avoid technical jargon unless necessary.
    - Keep the answer brief and decision-oriented.
    """,
    "analytical": 
    """
    Respond in an Analytical Explanation style.
    - Explain the main patterns clearly.
    - Compare relevant groups or variables where useful.
    - Balance clarity and detail.
    - Keep the answer grounded in the dataset.
    - Do not overstate conclusions.
    """,
    "technical": """
    Respond in a Technical Interpretation style.
    - Refer explicitly to dataset variables where relevant.
    - Emphasize associations, not causality.
    - Mention limitations or ambiguity when appropriate.
    - Use more precise analytical language.
    - Keep the answer dataset-grounded and method-aware.
    """
}

## Define the evaluation questions

In [6]:
questions = [
    "Which job roles appear to have the highest burnout risk?",
    "How does AI tool usage relate to burnout risk in this dataset?",
    "Show employees with high burnout risk and low productivity, and summarize what they have in common.",
    "What workplace factors seem most associated with burnout in this dataset?"
]

## Initialize the model

In [7]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if not anthropic_key:
    raise ValueError("ANTHROPIC_API_KEY is not set in your environment.")

chat_model = ChatAnthropic(model="claude-sonnet-4-0")